In [1]:
import json

with open("../data/match.json") as f:
    data = json.load(f)

data

[{'Home team': 'Man United',
  'Away team': 'Birmingham',
  'Final score': '1-0',
  'Ground': 'Old Trafford',
  'goals': [{'minute': '25',
    'player': 'Carlos Tevez',
    'team': 'Man United',
    'evidence': 'Tevez scored the only goal as Manchester United secured a narrow victory... Tevez... calmly slotted home the winner on 25 minutes.'}],
  'red_cards': [],
  'home_manager': 'Sir Alex Ferguson',
  'home_manager_evidence': 'United manager Sir Alex Ferguson, sat in the stands to serve out his two-match touchline ban...',
  'away_manager': 'Alex McLeish',
  'away_manager_evidence': 'Birmingham boss Alex McLeish - once a protege of Ferguson in their days together at Aberdeen...',
  'man_of_the_match': 'Carlos Tevez',
  'man_of_the_match_evidence': "BBC Sport Player Rater man of the match: Manchester United's Carlos Tevez 8.15"},
 {'Home team': 'Man United',
  'Away team': 'Hull',
  'Final score': '4-3',
  'Ground': 'Old Trafford',
  'goals': [{'minute': '3',
    'player': 'Cristiano 

In [2]:
# Code that produces first Excel spreadsheet (match level data)

import pandas as pd

match_rows = []
for match in data:
    match_rows.append({
        "home team": match["Home team"],
        "home manager": match["home_manager"],
        "home score": match["Final score"].split("-")[0],
        "away team": match["Away team"],
        "away manager": match["away_manager"],
        "away score": match["Final score"].split("-")[1],
        "stadium": match["Ground"]
    })
matches_df = pd.DataFrame(match_rows)
matches_df

,home team,home manager,home score,away team,away manager,away score,stadium
0,Man United,Sir Alex Ferguson,1,Birmingham,Alex McLeish,0,Old Trafford
1,Man United,Sir Alex Ferguson,4,Hull,Phil Brown,3,Old Trafford
2,Aston Villa,Remi Garde,2,Wycombe,Gareth Ainsworth,0,Villa Park
3,Man United,Jose Mourinho,1,Celta Vigo,,1,Old Trafford
4,Man United,Jose Mourinho,1,Bournemouth,Eddie Howe,0,Old Trafford
5,Man United,Ole Gunnar Solskjaer,3,Brighton,Graham Potter,1,Old Trafford
6,Man United,Ole Gunnar Solskjær,2,Aston Villa,Dean Smith,2,Old Trafford
7,Leicester,Brendan Rodgers,1,Norwich,Daniel Farke,1,King Power Stadium
8,Man United,Ole Gunnar Solskjær,3,Watford,Nigel Pearson,0,Old Trafford
9,Norwich,Dean Smith,0,Man United,Ralf Rangnick,1,Carrow Road


In [3]:
def summarise_goals(goals):
    return ", ".join([
        f"{g['player']} ({g['minute']})"
        for g in goals
    ])

matches_df["goals_summary"] = [
    summarise_goals(match.get("goals", []))
    for match in data
]

matches_df

,home team,home manager,home score,away team,away manager,away score,stadium,goals_summary
0,Man United,Sir Alex Ferguson,1,Birmingham,Alex McLeish,0,Old Trafford,Carlos Tevez (25)
1,Man United,Sir Alex Ferguson,4,Hull,Phil Brown,3,Old Trafford,"Cristiano Ronaldo (3), Michael Carrick (29), C..."
2,Aston Villa,Remi Garde,2,Wycombe,Gareth Ainsworth,0,Villa Park,"Ciaran Clark (75'), Idrissa Gueye (90')"
3,Man United,Jose Mourinho,1,Celta Vigo,,1,Old Trafford,Marcus Rashford (67)
4,Man United,Jose Mourinho,1,Bournemouth,Eddie Howe,0,Old Trafford,Romelu Lukaku (25)
5,Man United,Ole Gunnar Solskjaer,3,Brighton,Graham Potter,1,Old Trafford,"Davy Propper (19), Andreas Pereira (), Marcus ..."
6,Man United,Ole Gunnar Solskjær,2,Aston Villa,Dean Smith,2,Old Trafford,"Jack Grealish (N/A), Marcus Rashford (N/A), Vi..."
7,Leicester,Brendan Rodgers,1,Norwich,Daniel Farke,1,King Power Stadium,"Teemu Pukki (25th), Tim Krul (37th)"
8,Man United,Ole Gunnar Solskjær,3,Watford,Nigel Pearson,0,Old Trafford,"Bruno Fernandes (unknown), Anthony Martial (un..."
9,Norwich,Dean Smith,0,Man United,Ralf Rangnick,1,Carrow Road,Cristiano Ronaldo (75)


In [4]:
from collections import Counter

goal_scorers = []
for match in data:
    for goal in match.get("goals", []):
        goal_scorers.append(goal["player"])

counts = Counter(goal_scorers)

ranking_df = (
    pd.DataFrame(
        counts.items(),
        columns=["Player", "Goals Seen Live"]
    )
    .sort_values(
        ["Goals Seen Live", "Player"],
        ascending=[False, True]
    )
)

ranking_df["Rank"] = (
    ranking_df["Goals Seen Live"]
    .rank(method="min", ascending=False)
    .astype(int)
)

ranking_df = ranking_df[
    ["Rank", "Player", "Goals Seen Live"]
].reset_index(drop=True)

ranking_df

,Rank,Player,Goals Seen Live
0,1,Marcus Rashford,5
1,2,Bruno Fernandes,4
2,3,Christopher Nkunku,3
3,3,Cristiano Ronaldo,3
4,5,Anthony Martial,2
5,5,Bryan Mbeumo,2
6,5,Bukayo Saka,2
7,5,Cole Palmer,2
8,5,Declan Rice,2
9,5,Matheus Cunha,2


In [5]:
with pd.ExcelWriter("../data/enriched_matches.xlsx", engine="openpyxl") as writer:
    matches_df.to_excel(writer, sheet_name="Matches", index=False)
    ranking_df.to_excel(writer, sheet_name="Top Scorers", index=False)